# Part 2 — Stream Stack

This notebook provisions the streaming environment: a short-lived VM that pulls the scoring image from [Part 1](tutorial_2.ipynb), replays transactions through Pub/Sub, scores them in near real time, and writes a fraud report. It assumes the shared **data stack** from [tutorial_1.ipynb](tutorial_1.ipynb) is already deployed and trained model exists.

## Contents

1. [Stream Stack](#10-stream-stack)
    - [1.1 Pub/Sub topic and subscription](#11-pubsub-topic-and-subscription)
    - [1.2 Publisher and consumer](#12-publisher-and-consumer)
    - [1.3 Evaluation](#13-evaluation)

← Back to [tutorial_2.ipynb](tutorial_2.ipynb) · ← Shared setup in [tutorial_1.ipynb](tutorial_1.ipynb)

## 1.0 Stream Stack
The stream stack creates a short-lived private network and a VM whose job is to *serve* predictions rather than produce them. It **simulates a production deployment** in which the trained model continuously consumes credit card transactions and flags suspected fraud in near real time.

In a real deployment the consumer would run indefinitely, since transactions arrive for as long as cards are being swiped. Since this is a tutorial we need a way to end the run and collect a report, so the publisher emits a **sentinel** message after the last CSV row. The sentinel is a tutorial-only artefact that tells *us* the simulation is over; a production consumer would not have or expect one.

The pipeline can be broken down to 6 steps, shown in figure 1 below:

1. **Boot** — Terraform creates the Pub/Sub topic and subscription, then the stream VM; `startup.sh` begins running.
2. **Pull** — `startup.sh` downloads the streaming CSV from GCS and pulls `fraud-scoring:v5` from Artifact Registry.
3. **Start consumer** — the scoring container starts in the background and subscribes to the Pub/Sub subscription. In production this is the steady state, and the consumer would stay running from here onwards.
4. **Publish** — the publisher replays the CSV rows as JSON messages at ~25 msg/s (standing in for live card swipes) and closes with the **sentinel** that ends the simulation.
5. **Score** — the consumer pulls messages in batches and runs them through the saved `PipelineModel`, flagging any rows the model predicts as fraud.
6. **Report** — once the sentinel arrives, the consumer writes `fraud_report.json` and `startup.sh` uploads it to the data stack's GCS bucket. This reporting step only exists because the simulation has an end. A production deployment would push flagged frauds to a downstream system instead.

<p align="center">
  <img src="images/stream_pipeline.png" width="600"><br>
  Fig 1. Stream Stack Pipeline
</p>

The stream VM is ephemeral in the same sense as the train VM: the topic, subscription, scoring image, and report all live outside it, so it can be destroyed once the report lands in GCS.

The networking layer is an identical copy of §1.1 in [tutorial_2.ipynb](tutorial_2.ipynb) (private VPC + Cloud NAT + IAP-only SSH) and is not repeated here. As in the train stack, the whole run is orchestrated by a single `startup.sh` rendered into the VM at boot via Terraform's `templatefile()`.

### 1.1 Pub/Sub topic and subscription

Pub/Sub is GCP's managed message bus. A **topic** is the named channel publishers write to; a **subscription** is a named backlog that retains messages until a consumer acks them. The stream stack creates one of each:

```hcl
# infra/terraform/stream/main.tf (abridged)
resource "google_pubsub_topic" "transactions" {
  name = var.pubsub_topic_id
}

resource "google_pubsub_subscription" "transactions_sub" {
  name                 = var.pubsub_subscription_id
  topic                = google_pubsub_topic.transactions.name
  ack_deadline_seconds = 60
}
```

If the consumer does not ack within 60s, Pub/Sub assumes the message was lost and **redelivers** it. This is how the service guarantees **at-least-once** delivery: a message is only removed from the backlog once the consumer explicitly confirms it was processed.

### 1.2 Publisher and consumer

The **publisher** ([scripts/publish_transactions.py](scripts/publish_transactions.py)) runs directly on the VM host. It reads the streaming subset CSV row-by-row and publishes each row as a JSON message at 25 msg/s, standing in for live card swipes. After the last row it emits a **sentinel** that marks the end of the simulation. A real publisher would simply keep streaming forever and would not have a sentinel at all.

The **consumer** ([scripts/score_stream.py](scripts/score_stream.py)) runs inside the `fraud-scoring:v5` container, which already contains Spark and the saved `PipelineModel`. Its job is to keep the pipeline hot in memory and score each Pub/Sub message as soon as it arrives, so a flagged transaction shows up roughly a second after it was published.

The consumer is a **plain Python pull loop** running on two threads:

- A **background thread** listens to Pub/Sub and drops each new transaction into a shared in-memory buffer.
- The **main thread** wakes every second (or sooner, if the buffer hits 50 rows), hands the batch to Spark, and adds rows where `prediction == 1.0` to a running list of flagged frauds.

The skeleton looks like this:

```python
# scripts/score_stream.py (abridged)
state = {"buffer": [], "frauds": [], "done": False}
lock  = threading.Lock()

def callback(message):                                  # background thread
    payload = json.loads(message.data.decode("utf-8"))
    if payload.get("__sentinel") == "DONE":
        with lock: state["done"] = True
    else:
        with lock: state["buffer"].append(payload)
    message.ack()

subscriber.subscribe(subscription_path, callback=callback)

last_drain = time.time()
while True:                                             # main thread
    with lock:
        buffered = len(state["buffer"])
        done     = state["done"]

    if buffered >= 50 or (buffered and time.time() - last_drain > 1.0):
        with lock:
            batch, state["buffer"] = state["buffer"], []
        scored  = model.transform(spark.createDataFrame(batch, schema=schema))
        flagged = scored.filter(scored.prediction == 1.0).collect()
        with lock: state["frauds"].extend(flagged)
        last_drain = time.time()

    if done and not buffered:
        break
    time.sleep(0.2)
```

In production this loop would run forever and push flagged frauds to another system (a Pub/Sub topic, a database, or an alerting service). Here the sentinel ends things: on arrival, the consumer finishes the current buffer, writes `fraud_report.json` with the message counts, flagged samples, and elapsed time, and exits. `startup.sh` uploads the report to GCS and the VM's work is done.

> **Deploy now.** To spin the stream stack up as you read, run `terraform init && terraform apply` in `infra/terraform/stream/`. The VM pulls the scoring image, replays transactions through Pub/Sub, and writes `fraud_report.json` to GCS, then can be destroyed with `terraform destroy`.

### 1.3 Evaluation

Sample numbers from a real run of the deployed pipeline:

| Metric | Value |
|---|---|
| Messages published | 3,000 + 1 sentinel |
| Messages seen by consumer | 3,000 (no loss) |
| Frauds flagged (raw count) | 6 |
| Frauds flagged (deduped on `Time`) | 1 |
| Elapsed seconds | 114.4 |
| Training AUC-ROC / AUC-PR | 0.9645 / 0.7149 |

The six-vs-one gap is the at-least-once redelivery: the same transaction was scored multiple times across redelivered messages, and the `Time` dedup collapses them back to the single underlying fraud.

---

**Teardown.** Once `fraud_report.json` lands in GCS, the stream VM's work is done. Run `terraform destroy` in [infra/terraform/stream/](infra/terraform/stream/), or use [scripts/destroy_everything.sh](scripts/destroy_everything.sh) / [scripts/destroy_everything.ps1](scripts/destroy_everything.ps1) to tear down stream + train + data stacks in the right order.